# Within-Dataset Benchmark: Ridge, Random Forest, LightGBM, GraphDRP, SimpleLinearNN

This notebook runs the first benchmark stage using the official IMPROVE within-dataset splits. It starts with five models:

- `ridge`
- `random_forest`
- `lightgbm`
- `graphdrp`
- `simple_linear_nn`

The code is kept in `within_dataset_4models.py` so the model registry can be extended later without turning the notebook into a maze.

In [1]:
from pathlib import Path
import importlib
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "new_notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# GraphDRP official preprocessing uses improvelib.statics.LINCS_SYMBOL.
# In the demo env this should already import; the path add below helps local runs.
IMPROVE_DIR = Path(os.environ.get("IMPROVE_DIR", ROOT.parent / "IMPROVE"))
if IMPROVE_DIR.exists() and str(IMPROVE_DIR) not in sys.path:
    sys.path.insert(0, str(IMPROVE_DIR))

try:
    from improvelib.statics import LINCS_SYMBOL
except ImportError as exc:
    raise ImportError(
        "This notebook expects the demo env with improvelib installed. "
        "GraphDRP official preprocessing requires improvelib.statics.LINCS_SYMBOL."
    ) from exc

import within_dataset_4models
importlib.reload(within_dataset_4models)
from within_dataset_4models import (
    CELL_ID_COL,
    get_lincs_symbol_list,
    make_default_config,
    read_gene_expression,
    read_response,
    response_for_split,
    run_within_dataset_benchmark,
    select_gene_columns,
    summarize_results,
)

cfg = make_default_config(ROOT)
cfg


BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest', 'lightgbm', 'graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Config

Default is a quick smoke test on `CCLE`, fold `0`. For the full paper-style within-dataset run, switch to all five datasets and ten folds.

In [2]:
# Smoke test / current experiment
cfg.datasets = ["CCLE"]
cfg.folds = list(range(3))

# Full paper-style within-dataset benchmark. Uncomment when ready.
# cfg.datasets = ["gCSI", "CCLE", "GDSCv2", "GDSCv1", "CTRPv2"]
# cfg.folds = list(range(10))

# cfg.models = ["ridge", "random_forest", "lightgbm", "graphdrp", "simple_linear_nn"]
cfg.models = ["graphdrp", "simple_linear_nn"]


# Required for official GraphDRP/SimpleLinearNN/RF-style preprocessing.
# If this is True and improvelib is unavailable, the runner raises instead of silently using 512 top-variance genes.
cfg.use_lincs_symbol_genes = True

# Mordred can still be capped for tabular models; top_ge_features is ignored while use_lincs_symbol_genes=True.
cfg.top_ge_features = 512
cfg.top_mordred_features = 512

cfg.graphdrp_epochs = 150
cfg.graphdrp_patience = 20
cfg.simple_nn_epochs = 300
cfg.simple_nn_patience = 50
cfg.simple_nn_model = "default"

# Use these for a very quick debug run, then set back to None.
cfg.max_train_rows = None
cfg.max_eval_rows = None

cfg


BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0, 1, 2], models=['graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Validate GraphDRP Preprocessing

GraphDRP official `develop` uses `[['subset', 'LINCS_SYMBOL'], ['scale', 'std']]`. This check must show `958` mapped gene-expression columns on the current CSA data; `512` means the notebook is using the old top-variance fallback.

In [3]:
lincs_symbols = get_lincs_symbol_list(required=True)
response_check = read_response(cfg)
gene_expression_check = read_gene_expression(cfg)
train_check = response_for_split(cfg, response_check, cfg.datasets[0], cfg.folds[0], "train")
ge_cols_check = select_gene_columns(cfg, gene_expression_check, train_check[CELL_ID_COL].unique())

print(f"improvelib LINCS_SYMBOL: {len(lincs_symbols)}")
print(f"Mapped gene-expression columns: {len(ge_cols_check)}")
assert len(lincs_symbols) == 976
assert len(ge_cols_check) == 958, (
    f"Expected 958 mapped LINCS gene-expression columns, got {len(ge_cols_check)}. "
    "If this says 512, reload within_dataset_4models.py and make sure the demo env imports improvelib."
)


improvelib LINCS_SYMBOL: 976
Mapped gene-expression columns: 958


## Run Benchmark

GraphDRP requires `torch`, `torch-geometric`, and `rdkit`. SimpleLinearNN requires `torch`. If dependencies are missing, the runner records the model as skipped instead of stopping the whole benchmark.

In [4]:
results = run_within_dataset_benchmark(cfg)
display(results)

Dataset=CCLE | fold=0
Rows usable: train=7,616, val=952, test=951 | tabular features=1,470
  Training graphdrp
    val: n=952 RMSE=0.0772 MAE=0.0605 R2=0.7498 Pearson=0.8797
    test: n=951 RMSE=0.0789 MAE=0.0621 R2=0.7714 Pearson=0.8874
  Training simple_linear_nn
    val: n=952 RMSE=0.0778 MAE=0.0610 R2=0.7459 Pearson=0.8649
    test: n=951 RMSE=0.0789 MAE=0.0618 R2=0.7712 Pearson=0.8784
Dataset=CCLE | fold=1
Rows usable: train=7,616, val=951, test=952 | tabular features=1,470
  Training graphdrp


KeyboardInterrupt: 

## Summary

In [ ]:
summary = summarize_results(results, cfg.out_dir)
display(summary)